# Speculative Decoding

# Helper Functions

In [14]:
import torch
from gpt_model import GPTModel

In [10]:
# detect what hardware is available
# basically, detect the best fastest available processor
# on our computer so that your GPT model doesn't run at 
# a snail's pace on the CPU if a better option exists.

# check for an NVIDIA graphics card with CUDA drivers installed
# GOLD STANDARD

def pick_device():
    if torch.cuda.is_available():
        device = torch.device("cuda")
    # check for apple GPUs built into M1, M2 or M3 chips
    elif torch.backends.mps.is_available():
        # Only use GPU on a mac if the PyTorch supports it
        major, minor = map(int, torch.__version__.split(".")[:2])
        if (major, minor) >= (2, 9):
            device = torch.device("mps")
        else:
            device = torch.device("cpu")
    # fallback to CPU
    else:
        device = torch.device("cpu")

    return device

In [11]:
# Helper to download open weights for GPT-2.0, 
# already in PyTorch format.

import requests
import os

def download_weights_from_openai(filename):
    url = f"https://huggingface.co/rasbt/gpt2-from-scratch-pytorch/resolve/main/{filename}"
    
    if not os.path.exists(filename):
        print(f"Downloading {filename}...")
        # 'verify=False' bypasses the SSL check entirely
        response = requests.get(url, verify=False)
        with open(filename, "wb") as f:
            f.write(response.content)
        print(f"Successfully downloaded to {filename}!")
    else:
        print(f"{filename} already exists.")

In [19]:
# map the downloaded weights to my own class for GPT-2.0


def load_model(weights_path, config, device):
    model = GPTModel(config)

    # Load the state dictionary
    pretrained_state_dict = torch.load(
        weights_path, map_location=device, weights_only=True
    )

    # Create a new dict with updated keys
    new_state_dict = {}
    for key, value in pretrained_state_dict.items():
        new_key = key
        # Map attributes: 'trf_blocks' -> 'transformer_blocks',
        # 'att' -> 'mha', 'norm1' -> 'layerNorm1', 'norm2' -> 'layerNorm2'
        new_key = new_key.replace("tok_emb", "sem_emb")
        new_key = new_key.replace("trf_blocks", "transformer_blocks")
        new_key = new_key.replace(".att.", ".mha.")
        new_key = new_key.replace(".norm1.", ".layerNorm1.")
        new_key = new_key.replace(".norm2.", ".layerNorm2.")

        new_state_dict[new_key] = value

    # 4. Load the remapped state dict
    model.load_state_dict(new_state_dict)
    print("successfully mapped the model weights to our architecture")

    return model

# Setup the draft and target models

## Load the draft model (GPT-2.0 124M)

In [6]:
GPT_CONFIG_124M_OPEN_AI = {
    "vocab_size": 50257,   # Vocabulary size
    "context_length": 1024, # Shortened context length (orig: 1024)
    "emb_dim": 768,        # Embedding dimension
    "n_heads": 12,         # Number of attention heads
    "n_layers": 12,        # Number of layers
    "drop_rate": 0.1,      # Dropout rate
    "qkv_bias": True,      # Query-key-value bias
    "weight_tying": False
}

In [20]:
path_to_draft_model = "gpt2-small-124M.pth"

download_weights_from_openai(filename=path_to_draft_model)
model = load_model(
    weights_path=path_to_draft_model,
    config=GPT_CONFIG_124M_OPEN_AI,
    device=pick_device(),
)

gpt2-small-124M.pth already exists.
successfully mapped the model weights to our architecture


## Load the Target Model (GPT-2.0 1558M)

In [ ]:
_GPT_CONFIG_1558M_OPEN_AI = {
    "vocab_size": 50257,   # Vocabulary size
    "context_length": 1024, # 
    "emb_dim": 1600,        # Embedding dimension
    "n_heads": 25,         # Number of attention heads
    "n_layers": 48,        # Number of transformer layers
    "drop_rate": 0.1,      # Dropout rate
    "qkv_bias": True,      # Query-key-value bias
    "weight_tying": False
}

In [27]:
GPT_CONFIG_355M_OPEN_AI = {
    "vocab_size": 50257,   # Vocabulary size
    "context_length": 1024, # 
    "emb_dim": 1024,        # Embedding dimension
    "n_heads": 16,         # Number of attention heads
    "n_layers": 24,        # Number of transformer layers
    "drop_rate": 0.1,      # Dropout rate
    "qkv_bias": True,      # Query-key-value bias
    "weight_tying": False
}

In [28]:
path_to_target_model = "gpt2-medium-355M.pth"

download_weights_from_openai(filename=path_to_target_model)
model = load_model(
    weights_path=path_to_target_model,
    config=GPT_CONFIG_355M_OPEN_AI,
    device=pick_device(),
)

/Users/vidhant/Downloads/llm-from-scratch/.venv/lib/python3.10/site-packages/urllib3/connectionpool.py:1110: InsecureRequestWarning: Unverified HTTPS request is being made to host 'huggingface.co'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
/Users/vidhant/Downloads/llm-from-scratch/.venv/lib/python3.10/site-packages/urllib3/connectionpool.py:1110: InsecureRequestWarning: Unverified HTTPS request is being made to host 'cas-bridge.xethub.hf.co'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


Successfully downloaded to gpt2-medium-355M.pth!


RuntimeError: Error(s) in loading state_dict for GPTModel:
	Missing key(s) in state_dict: "transformer_blocks.24.layerNorm1.scale", "transformer_blocks.24.layerNorm1.shift", "transformer_blocks.24.mha.mask", "transformer_blocks.24.mha.W_query.weight", "transformer_blocks.24.mha.W_query.bias", "transformer_blocks.24.mha.W_key.weight", "transformer_blocks.24.mha.W_key.bias", "transformer_blocks.24.mha.W_value.weight", "transformer_blocks.24.mha.W_value.bias", "transformer_blocks.24.mha.out_proj.weight", "transformer_blocks.24.mha.out_proj.bias", "transformer_blocks.24.layerNorm2.scale", "transformer_blocks.24.layerNorm2.shift", "transformer_blocks.24.ff.layers.0.weight", "transformer_blocks.24.ff.layers.0.bias", "transformer_blocks.24.ff.layers.2.weight", "transformer_blocks.24.ff.layers.2.bias", "transformer_blocks.25.layerNorm1.scale", "transformer_blocks.25.layerNorm1.shift", "transformer_blocks.25.mha.mask", "transformer_blocks.25.mha.W_query.weight", "transformer_blocks.25.mha.W_que